# ⏱️ TP-4 : Prédiction de Séries Temporelles avec LSTM

**Objectif** : Prédire la consommation électrique avec des réseaux LSTM.

**Compétences** :
- Préparation de données temporelles
- Fenêtres glissantes (windowing)
- Architecture LSTM avec Keras
- Early stopping et régularisation

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

# Reproductibilité
np.random.seed(42)
tf.random.set_seed(42)

print(f"✅ TensorFlow version : {tf.__version__}")

In [ ]:
# Génération de données synthétiques (consommation électrique)
# En pratique, remplacez par vos données réelles

def generate_energy_data(n_days=365*2):
    """Génère des données de consommation électrique simulées"""
    hours = np.arange(n_days * 24)
    
    # Tendance
    trend = 0.001 * hours
    
    # Saisonnalité journalière
    daily = 10 * np.sin(2 * np.pi * hours / 24)
    
    # Saisonnalité hebdomadaire
    weekly = 5 * np.sin(2 * np.pi * hours / (24 * 7))
    
    # Saisonnalité annuelle
    yearly = 15 * np.sin(2 * np.pi * hours / (24 * 365))
    
    # Bruit
    noise = np.random.normal(0, 3, len(hours))
    
    # Consommation totale
    consumption = 50 + trend + daily + weekly + yearly + noise
    consumption = np.maximum(consumption, 0)  # Pas de valeurs négatives
    
    return consumption

# Génération des données
data = generate_energy_data(n_days=730)  # 2 ans de données

# Création du DataFrame
dates = pd.date_range(start='2022-01-01', periods=len(data), freq='H')
df = pd.DataFrame({'consumption': data}, index=dates)

print(f"📊 Période : {df.index[0]} à {df.index[-1]}")
print(f"📊 Total : {len(df)} heures de données")
df.head()

In [ ]:
# Visualisation des données
fig, axes = plt.subplots(3, 1, figsize=(15, 10))

# Vue complète
axes[0].plot(df.index, df['consumption'], alpha=0.7)
axes[0].set_title('Consommation électrique - Vue complète (2 ans)')
axes[0].set_ylabel('kWh')

# Vue d'une semaine
one_week = df.iloc[:24*7]
axes[1].plot(one_week.index, one_week['consumption'], marker='o')
axes[1].set_title('Consommation - Vue hebdomadaire')
axes[1].set_ylabel('kWh')

# Vue d'une journée
one_day = df.iloc[:24]
axes[2].plot(one_day.index.hour, one_day['consumption'], marker='o')
axes[2].set_title('Consommation - Vue journalière')
axes[2].set_xlabel('Heure')
axes[2].set_ylabel('kWh')

plt.tight_layout()
plt.show()

In [ ]:
# Feature Engineering temporel
df['hour'] = df.index.hour
df['day_of_week'] = df.index.dayofweek
df['month'] = df.index.month
df['is_weekend'] = (df['day_of_week'] >= 5).astype(int)

# Lags (valeurs précédentes)
for lag in [1, 2, 3, 24, 48]:
    df[f'lag_{lag}'] = df['consumption'].shift(lag)

# Rolling statistics
df['rolling_mean_24'] = df['consumption'].rolling(window=24).mean()
df['rolling_std_24'] = df['consumption'].rolling(window=24).std()

# Suppression des NaN
df = df.dropna()

print(f"✅ Features créées : {df.shape[1]} colonnes")
df.head()

In [ ]:
# Préparation des séquences pour LSTM
def create_sequences(data, target_col, sequence_length=24):
    """
    Crée des séquences pour LSTM
    data : DataFrame avec features
    target_col : nom de la colonne cible
    sequence_length : longueur de la séquence (ex: 24 heures)
    """
    X, y = [], []
    values = data.values
    target_idx = data.columns.get_loc(target_col)
    
    for i in range(sequence_length, len(values)):
        X.append(values[i-sequence_length:i])
        y.append(values[i, target_idx])
    
    return np.array(X), np.array(y)

# Séparation train/test
train_size = int(len(df) * 0.8)
train_df = df.iloc[:train_size]
test_df = df.iloc[train_size:]

# Normalisation
scaler = MinMaxScaler()
train_scaled = scaler.fit_transform(train_df)
test_scaled = scaler.transform(test_df)

# Conversion en DataFrame pour garder les noms de colonnes
train_scaled = pd.DataFrame(train_scaled, columns=df.columns, index=train_df.index)
test_scaled = pd.DataFrame(test_scaled, columns=df.columns, index=test_df.index)

# Création des séquences
SEQUENCE_LENGTH = 24  # 24 heures d'historique

X_train, y_train = create_sequences(train_scaled, 'consumption', SEQUENCE_LENGTH)
X_test, y_test = create_sequences(test_scaled, 'consumption', SEQUENCE_LENGTH)

print(f"📊 X_train shape : {X_train.shape}")
print(f"📊 y_train shape : {y_train.shape}")
print(f"📊 X_test shape : {X_test.shape}")
print(f"📊 y_test shape : {y_test.shape}")

In [ ]:
# Construction du modèle LSTM
model = Sequential([
    LSTM(64, return_sequences=True, input_shape=(SEQUENCE_LENGTH, X_train.shape[2])),
    Dropout(0.2),
    LSTM(32, return_sequences=False),
    Dropout(0.2),
    Dense(16, activation='relu'),
    Dense(1)
])

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss='mse',
    metrics=['mae']
)

model.summary()

In [ ]:
# Callbacks
callbacks = [
    EarlyStopping(
        monitor='val_loss',
        patience=10,
        restore_best_weights=True
    ),
    ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=5,
        min_lr=1e-6
    )
]

# Entraînement
history = model.fit(
    X_train, y_train,
    epochs=100,
    batch_size=32,
    validation_split=0.2,
    callbacks=callbacks,
    verbose=1
)

In [ ]:
# Visualisation de l'entraînement
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Loss
axes[0].plot(history.history['loss'], label='Train')
axes[0].plot(history.history['val_loss'], label='Validation')
axes[0].set_title('Loss (MSE)')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].legend()

# MAE
axes[1].plot(history.history['mae'], label='Train')
axes[1].plot(history.history['val_mae'], label='Validation')
axes[1].set_title('MAE')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('MAE')
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
# Prédictions
y_pred = model.predict(X_test)

# Métriques (sur données normalisées)
mse = mean_squared_error(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mse)

print(f"📊 MSE  : {mse:.6f}")
print(f"📊 MAE  : {mae:.6f}")
print(f"📊 RMSE : {rmse:.6f}")

In [ ]:
# Visualisation des prédictions
plt.figure(figsize=(15, 6))

# Plot des 500 premières prédictions
n_plot = 500
plt.plot(y_test[:n_plot], label='Réel', alpha=0.8)
plt.plot(y_pred[:n_plot], label='Prédit', alpha=0.8)
plt.title(f'Prédictions LSTM - {n_plot} premiers points de test')
plt.xlabel('Temps')
plt.ylabel('Consommation (normalisée)')
plt.legend()
plt.show()

In [ ]:
# Scatter plot : Réel vs Prédit
plt.figure(figsize=(8, 8))
plt.scatter(y_test, y_pred, alpha=0.5)
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2)
plt.xlabel('Valeurs réelles')
plt.ylabel('Valeurs prédites')
plt.title('Réel vs Prédit')
plt.show()

## 🎓 Conclusion

Dans ce TP, nous avons :

1. ✅ **Généré** des données de consommation électrique avec patterns temporels
2. ✅ **Créé** des features temporelles (heure, jour, mois, lags, rolling stats)
3. ✅ **Préparé** les séquences pour LSTM avec windowing
4. ✅ **Construit** un modèle LSTM avec Dropout et Early Stopping
5. ✅ **Évalué** les performances sur l'ensemble de test

**Améliorations possibles** :
- Utiliser des données météo comme features externes
- Tester des architectures plus complexes (Bidirectional LSTM, GRU)
- Faire du multi-step forecasting (prédire plusieurs heures en avance)